# 环节 03 · 位置编码（配套 Notebook）

> 配套长文：[环节03-位置编码详解.md](./环节03-位置编码详解.md)
> 定位：把长文里的**时针类比 + 两组关键实验 + 频率表**全部跑成数字。纯 Python 标准库，零依赖。

**怎么跑**：逐格 `Shift+Enter`；后面的格子依赖前面已执行的变量。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 顺序无感 | §1 | Attention 是排列等变（permutation equivariance） |
| §2 绝对位置的两个硬伤 | §2 | "相加"混淆信息 + 越界发不出卡 |
| §3 旋转工具 | §4.1 | `k@45° = (0.1414, 0.8485)` |
| §4 两组关键实验 | §4.3 / §4.4 | 相对位置相同 → 分数相同；不同 → 分数不同 |
| §5 多档转速 | §5 | 单档会"撞车"，多档才分得清远近 |
| §6 真实 RoPE | §5 / §7.1 | `θ_k = 1/10000^(2k/d)` 的频率表与波长 |
| §7 长上下文外推 | §7.2 | PI / NTK / YaRN 在改什么 |


## 1. 为什么必须补"顺序"（长文 §1）

"猫追狗" 和 "狗追猫" 用的字完全一样。如果只给模型三张"内容卡"，它看到的两次是同一堆卡——因为注意力打分**只用内容坐标计算**，顺序变了分数不变。这个性质叫**排列等变（permutation equivariance）**。

下面用三个词的玩具坐标验证这一点：


In [ ]:
import math


def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def scores(X):
    """注意力分数矩阵：只用内容坐标算，全程不碰位置。"""
    d = len(X[0])
    return [[dot(X[i], X[j]) / math.sqrt(d) for j in range(len(X))] for i in range(len(X))]


NAMES = ["猫", "追", "狗"]
X = [
    [1.0, 0.0, 0.5],     # 猫
    [0.0, 1.0, 0.5],     # 追
    [1.0, 1.0, 0.0],     # 狗
]


def show(title, names, X):
    print(f"\n{title}（行 = 谁在看，列 = 被看谁）")
    print("        " + "".join(f"{n:>8}" for n in names))
    S = scores(X)
    for i, n in enumerate(names):
        print(f"{n:>6}  " + "".join(f"{v:>8.4f}" for v in S[i]))
    return S


S_normal = show("原句：猫 追 狗", NAMES, X)
order = [2, 1, 0]                                   # 换成 狗 追 猫
S_swapped = show("打乱：狗 追 猫", [NAMES[i] for i in order], [X[i] for i in order])

print(f"\n两次的“猫·追”分数：{S_normal[0][1]:.4f} 和 {S_swapped[2][1]:.4f} —— 一模一样")
print("→ 整个分数矩阵只是被“跟着搬了位置”，没有任何一个词对的关系发生变化。")
print("  模型因此分不清“猫追狗”和“狗追猫” —— 必须额外把位置信息喂进去。")


## 2. 老办法的两个硬伤（长文 §2）

把位置也做成一张"卡"，和内容坐标**相加**：

```
第1个字：猫坐标(1.0,0.5) + 位置1的卡(0.0,0.1) → (1.0,0.6)
第2个字：狗坐标(0.9,0.7) + 位置2的卡(0.0,0.2) → (0.9,0.9)
```

两个硬伤：① 词义和位置"混成一锅粥"；② **没见过的位置发不出卡**（外推差）。第 ② 点在代码里最直观：


In [ ]:
MAX_TRAIN_LEN = 8                                     # 玩具：训练时最长见过 8 个位置
LEARNED_CARDS = [f"第{i}号卡（训练学到的）" for i in range(MAX_TRAIN_LEN)]


def learned_position_card(pos):
    """可学习绝对位置：查表。表只到 MAX_TRAIN_LEN，超出就没有卡了。"""
    return LEARNED_CARDS[pos] if pos < MAX_TRAIN_LEN else "❌ 没有这张卡（训练时没见过）"


def rope_phase(pos, theta=0.7):
    """RoPE：位置只是"转多少弧度"，是个公式 —— 任意 pos 都算得出来。"""
    return f"转 {pos * theta:.2f} rad（cos={math.cos(pos * theta):+.3f}）"


print("位置     可学习绝对位置                    RoPE")
print("-" * 74)
for pos in (0, 7, 8, 100, 10000):
    print(f"{pos:>6}   {learned_position_card(pos):<32} {rope_phase(pos)}")

print("\n→ 表只到 8（= 训练长度），第 9 号之后全是“没卡”；")
print("  RoPE 的“卡”是算出来的，位置多大都有定义 —— 结构上限 ≈ 无限（长文 §7.2）。")


## 3. 旋转这件小工具（长文 §4.1）

向量 `(x, y)` 转 φ 角：

```
新x = x·cosφ − y·sinφ
新y = x·sinφ + y·cosφ
```

把长文 §4.1 的例子算出来（`k = (0.7, 0.5)` 转 45°）：


In [ ]:
def rotate(v, deg):
    """把二维向量逆时针旋转 deg 度。"""
    a = math.radians(deg)
    c, s = math.cos(a), math.sin(a)
    return (v[0] * c - v[1] * s, v[0] * s + v[1] * c)


for deg in (0, 45, 90, 180):
    print(f"  (0.7, 0.5) 转 {deg:>3}° → ({rotate((0.7, 0.5), deg)[0]:+.4f}, "
          f"{rotate((0.7, 0.5), deg)[1]:+.4f})")

print("\n转 45° 的结果 = (0.1414, 0.8485)，与长文 §4.1 手算一致。")
print("注意旋转不改变向量长度（只是指针拨了个角度）：")
for deg in (0, 45, 90, 180):
    v = rotate((0.7, 0.5), deg)
    print(f"  {deg:>3}° 后长度 = {math.sqrt(v[0] ** 2 + v[1] ** 2):.4f}")


## 4. 两组关键实验（长文 §4.3 / §4.4）

设定：**第 m 个位置 → 转 m × 45°**。`q = (1, 0)`、`k = (0.7, 0.5)`，分数 = 旋转后的点积。

- **实验 A**：绝对位置不同、相对位置相同 → 分数必须相同；
- **实验 B**：相对位置不同 → 分数必须不同（模型能分辨远近）。


In [ ]:
q, k = (1.0, 0.0), (0.7, 0.5)
STEP = 45


def rope_score(q, k, pos_q, pos_k, step=STEP):
    """q 转 pos_q × step 度、k 转 pos_k × step 度，再点积。"""
    return dot(rotate(q, pos_q * step), rotate(k, pos_k * step))


print("实验 A：绝对位置不同，相对位置相同")
print(f"  q@第2位 · k@第1位：{rope_score(q, k, 2, 1):.4f}")
print(f"  q@第5位 · k@第4位：{rope_score(q, k, 5, 4):.4f}   ← 仍是 0.8485")
print("  → k 都“紧挨在 q 前一位”，模型不在乎绝对第几号位。\n")

print("实验 B：相对位置不同")
for gap, note in [(0, "同位置"), (-1, "k 在 q 前 1 位"), (1, "k 在 q 后 1 位"), (2, "k 在 q 后 2 位")]:
    print(f"  gap={gap:>2}（{note:<12}）：{rope_score(q, k, 2, 2 + gap):+.4f}")
print("\n  → 同一对向量，只因为“隔了几格”不同，分数从 +0.85 一路变成 −0.50。")

print("\n把实验 A 做成通用验证（随机 q/k，任意绝对位置）：")
import random
random.seed(5)
q2 = [random.uniform(-1, 1) for _ in range(2)]
k2 = [random.uniform(-1, 1) for _ in range(2)]
ref = dot(rotate(q2, 0), rotate(k2, 1 * STEP))
for m, n in [(0, 1), (3, 4), (10, 11), (100, 101), (999, 1000)]:
    v = dot(rotate(q2, m * STEP), rotate(k2, n * STEP))
    print(f"  (m={m:>4}, n={n:>4}) 分数 {v:+.9f}   与 (0,1) 的差 {abs(v - ref):.1e}")
print("\n→ 数学根据：旋转矩阵是正交的（orthogonal），")
print("  (R_m q)·(R_n k) = q·(R_{n−m} k) —— 右边只剩位置差 n−m（长文 §4.3 进阶注记）。")


## 5. 为什么必须"多档转速"（长文 §5）

只有一根指针（每格 45°）时会撞车：转满一圈就回到原地。


In [ ]:
print("单档转速（每格 45°）的撞车：")
for pos in (0, 1, 8, 9, 16):
    v = rotate((1.0, 0.0), pos * 45)
    note = " ← 和位置 0 撞车！" if pos == 8 else (" ← 和位置 1 撞车！" if pos == 9 else "")
    print(f"  位置 {pos:>2}: 指针指向 ({v[0]:+.2f}, {v[1]:+.2f}){note}")
print("\n→ 每 8 步重复一次，这套编码只能区分 8 以内的相对距离。")
print("  把转速调慢能看清远方，但邻居之间几乎没角度差。一个指针解决不了两个矛盾 ——")
print("  所以 RoPE 把词向量拆成很多“小对”，每对一根转速不同的指针。")


In [ ]:
BASE = 10000.0            # RoPE 的 base，控制整体标尺（长文 §5 末）


def inv_freq(k, d, base=BASE):
    """第 k 对指针的角速度：θ_k = base^(-2k/d)。"""
    return base ** (-2.0 * k / d)


print("RoPE 的频率表（θ 越大转得越快 = 管紧邻；θ 越小转得越慢 = 管远距离）\n")
for d in (8, 128):
    print(f"d = {d}（每头维度）:")
    pairs = [(k, inv_freq(k, d), 2 * math.pi / inv_freq(k, d)) for k in range(d // 2)]
    show = pairs[:2] + [None] + pairs[-2:]
    for item in show:
        if item is None:
            print("      …")
            continue
        k, theta, turn = item
        tag = "最快（管紧邻）" if k == 0 else ("最慢（管远距离）" if k == d // 2 - 1 else "")
        print(f"    第 {k + 1:>2} 对: θ = {theta:.8f}   转一圈 ≈ {turn:>12,.0f} 步   {tag}")
    print()

print("→ 最快的档 6 步就转一圈（用来看紧挨着的字），最慢的档要几万步（用来看长距离呼应）；")
print("  不同 Attention 头各取所需：语法头用快指针、语义/主题头用慢指针。")


## 6. 真实 RoPE：把 Q / K 配对旋转（长文 §3.3 / §7.1）

标准实现：把 d 维向量按 `(x₀,x₁), (x₂,x₃), …` 配对，第 k 对转 `m · θ_k` 角度。**每层都要转一次**，且只转 Q、K，不转 V。


In [ ]:
def rope(vec, m, base=BASE):
    """给 d 维向量按位置 m 做 RoPE 旋转。"""
    d = len(vec)
    out = []
    for k in range(d // 2):
        ang = m * inv_freq(k, d, base)
        c, s = math.cos(ang), math.sin(ang)
        x, y = vec[2 * k], vec[2 * k + 1]
        out += [x * c - y * s, x * s + y * c]
    return out


random.seed(9)
d_demo = 8
qv = [random.uniform(-1, 1) for _ in range(d_demo)]
kv = [random.uniform(-1, 1) for _ in range(d_demo)]

print(f"d = {d_demo}，随机 q / k：")
print(f"  q@20  位 · k@17 位  = {dot(rope(qv, 20), rope(kv, 17)):+.6f}")
print(f"  q@120 位 · k@117位  = {dot(rope(qv, 120), rope(kv, 117)):+.6f}   ← 距离同为 3，分数相同")
print(f"  q@20  位 · k@19 位  = {dot(rope(qv, 20), rope(kv, 19)):+.6f}   ← 距离 1，分数不同")
print(f"  q@20  位 · k@21 位  = {dot(rope(qv, 20), rope(kv, 21)):+.6f}   ← 距离 1，但方向相反")

print("\n向量长度也没变（只是每对指针拨了角度）：")
print(f"  |q| = {math.sqrt(dot(qv, qv)):.6f}   |RoPE(q@12345)| = "
      f"{math.sqrt(dot(rope(qv, 12345), rope(qv, 12345))):.6f}")


## 7. 长上下文外推：PI / NTK / YaRN 在改什么（长文 §7.2）

长文的两句关键结论：

1. **结构上限 ≈ 无限**（RoPE 的位置就是"拨多少角度"，任意 m 都有定义）；
2. **能力上限 = L_train**（"隔多远该给多大注意力"是权重训出来的）。

所以超长上下文的问题不是"算不出来"，而是**新位置对应的相位区间，权重从没见过**。三种扩展手法的共同本质：**把位置缩小（等价于把频率调慢），让长位置重新落回训练见过的相位范围**。


In [ ]:
L_TRAIN = 2000
d_head = 128
theta_slow = inv_freq(d_head // 2 - 1, d_head)      # 最慢那档
max_phase_seen = L_TRAIN * theta_slow               # 训练期见过的最大相位

print(f"最慢档 θ = {theta_slow:.8f}")
print(f"训练长度 {L_TRAIN} 内见过的最大相位 = {max_phase_seen:.4f} rad "
      f"（转满一圈需要 {2 * math.pi / theta_slow:,.0f} 步）\n")

print("PI（位置插值）：把位置 m 换成 m/λ")
print(f"{'λ':>4} {'位置 8000 的相位':>18} {'是否落在训练范围':>18}")
for lam in (1, 2, 4, 8):
    phase = 8000 / lam * theta_slow
    ok = "✅ 在范围内" if phase <= max_phase_seen else "❌ 超出（权重没见过）"
    print(f"{lam:>4} {phase:>18.4f} {ok:>18}")

print("\n→ λ=4 时，位置 8000 的相位恰好回到 2000 以内的“舒适区”。")
print("  代价（长文 §7.2）：所有频率等比调慢后，快档也变慢 → 紧邻 token 的区分度下降。")
print("  于是有 NTK-aware（只压高频）与 YaRN（按波长分三档，转熟的不动、临界轻插、没转完的大插）。")


## 8. 自测（长文 §7.3）

| 问题 | 本 Notebook 的现场证据 |
|---|---|
| 为什么 Attention 天生不关心顺序？ | §1：打乱顺序后分数矩阵只是被搬了位置 |
| 可学习绝对位置的硬伤？ | §2：位置 9 之后"没有卡" |
| RoPE 的核心动作是什么？ | §3：旋转，且旋转不改变向量长度 |
| "相对位置"体现在哪？ | §4：`(R_m q)·(R_n k)` 只依赖 `n−m` |
| 为什么不能只用一根指针？ | §5：每 8 步撞一次车 |
| 为什么需要多档转速？ | §5：快档看邻居、慢档看远方 |
| "训练长度有限但想续更长"是什么意思？ | §7：结构上限无限，能力上限 = L_train；扩展手法都在重映射频率 |
| PI 的代价是什么？ | §7：全频率调慢 → 近邻区分度下降 |

**下一站**：[环节 04 · Attention](./环节04-Attention注意力详解.md)——旋转后的 Q/K 正式参与打分。
